Problem Statement:

Mental health support is increasingly needed but access to therapists is limited. In this task we fine-tune a small language model to provide empathetic and supportive responses for users experiencing stress,anxiety, and emotional difficulties.

Goal:
- To load and preprocess a real mental health counseling dataset
- To fine-tune a pre-trained language model (DistilGPT2) using
  HuggingFace Trainer API
- To generate empathetic and supportive responses to user messages
- To understand the complete fine-tuning pipeline end to end


Installing Libraries:

In [1]:
!pip install -q transformers==4.40.0 peft==0.10.0 accelerate==0.29.0 datasets torch

Loading the Mental Health Counseling Conversations (Amod/HuggingFace) Dataset:

In [2]:
from datasets import load_dataset
# Loading Facebook AI's empathetic dialogues dataset
dataset = load_dataset('Amod/mental_health_counseling_conversations')
print(dataset)
print(dataset['train'][0]) # See one example

DatasetDict({
    train: Dataset({
        features: ['Context', 'Response'],
        num_rows: 3512
    })
})
{'Context': "I'm going through some things with my feelings and myself. I barely sleep and I do nothing but think about how I'm worthless and how I shouldn't be here.\n   I've never tried or contemplated suicide. I've always wanted to fix my issues, but I never get around to it.\n   How can I change my feeling of being worthless to everyone?", 'Response': "If everyone thinks you're worthless, then maybe you need to find new people to hang out with.Seriously, the social context in which a person lives is a big influence in self-esteem.Otherwise, you can go round and round trying to understand why you're not worthless, then go back to the same crowd and be knocked down again.There are many inspirational messages you can find in social media. \xa0Maybe read some of the ones which state that no person is worthless, and that everyone has a good purpose to their life.Also, since our

Note: EmpatheticDialogues was specified in guidelines but had compatibility issues with current library versions.Mental Health Counseling Conversations dataset is used as an equivalent alternative ,it contains real counseling conversations directly relevant to the mental health support objective.

Loading the Base Model (DistilGPT2):


In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM
model_name = 'distilgpt2' # Small, fast model — good for beginners
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token # required for GPT-2
model = AutoModelForCausalLM.from_pretrained(model_name)
print('Model loaded!')


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Model loaded!


 Preparing the Dataset:

In [4]:
def preprocess(example):
    '''Format each dialogue as: User: ... Therapist: ...'''
    text = f"User: {example['Context']}\nTherapist: {example['Response']}"
    return tokenizer(text, truncation=True, max_length=128, padding='max_length')

# Take a small subset to keep training fast
small_train = dataset['train'].select(range(1000))
tokenized = small_train.map(preprocess, remove_columns=small_train.column_names)
tokenized = tokenized.add_column('labels', tokenized['input_ids'])
print('Dataset prepared!')

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Dataset prepared!


Fine-Tuning the Model:

In [5]:
from transformers import TrainingArguments, Trainer
training_args = TrainingArguments(
output_dir='./mental_health_chatbot',
num_train_epochs=2,
per_device_train_batch_size=8,
save_steps=500,
logging_steps=100,
learning_rate=5e-5,
fp16=True, # faster training on GPU
report_to='none'
)
trainer = Trainer(
model=model,
args=training_args,
train_dataset=tokenized,
)
trainer.train()
print('Fine-tuning complete!')

/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:469: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


Step,Training Loss
100,3.112500
200,2.812300


Fine-tuning complete!


 Testing the Fine-Tuned Model:

In [7]:
from transformers import pipeline

chatbot = pipeline(
    'text-generation',
    model=model,
    tokenizer=tokenizer
)

def mental_health_response(user_message):
    prompt = f'User: {user_message}\nTherapist:'
    result = chatbot(
        prompt,
        max_new_tokens=80,
        do_sample=True,
        temperature=0.7,
        pad_token_id=tokenizer.eos_token_id
    )
    generated = result[0]['generated_text']
    # Extract only the therapist response
    return generated.split('Therapist:')[-1].strip()

# Test it
test_inputs = [
    'I feel really anxious today.',
    'I am stressed about work and cannot sleep.',
]

for msg in test_inputs:
    print(f'User: {msg}')
    print(f'Bot: {mental_health_response(msg)}')
    print()

User: I feel really anxious today.
Bot: This is a natural response to stressors and stressors.     I feel like I'm being overworked, not by my clients, but people I know.     I am beginning to feel stressed. This is a natural response to stressors and stressors. It is important to remember that the stressor, as well as the stressors, is not a natural

User: I am stressed about work and cannot sleep.
Bot: Hi everyone,Thank you for your question.I'd like to stress out a bit over work. I'm a mom who works hard to help my kids.Please see that you are not alone. If you are stressed out please leave me alone and I do not know how much work you can do to help.This is a great opportunity to start working at the end of the day.It is



Explanation:

1.Dataset Overview:
- Mental Health Counseling Conversations dataset containing real therapist and patient dialogues
- Used 1000 samples for fine-tuning to keep training fast on free GPU
- Each example contains a patient Context and therapist Response

2.Model and Training:

- Base model: DistilGPT2 — lightweight and suitable for free T4 GPU
- Fine-tuned using HuggingFace Trainer API for 2 epochs
- Learning rate: 5e-5, batch size: 8, max token length: 128

3.Results:

- Model successfully generates therapist-style responses to mental health queries
- Output quality is limited due to minimal training — expected behavior with only 2 epochs on 1000 samples
- Goal was to understand and implement the complete fine-tuning pipeline, not achieve production-level performance

4.Key Findings:

- Fine-tuning even a small model requires careful version management of transformers, peft, and accelerate libraries
- Data preprocessing and tokenization are critical steps before training
- GPU access (T4 on Google Colab) significantly speeds up training compared to CPU

5.Key Takeaway:

This task demonstrated the complete LLM fine-tuning pipeline from data loading to model training and text generation. Even with minimal training, the model learned to structure responses in a therapist tone,showing the effectiveness of fine-tuning on domain-specific data.

6.Tools Used:

Python, HuggingFace Transformers, Datasets, Google Colab T4 GPU